# Cardio Catch Disease — 02. Exploratory Data Analysis

> **Hands-On ML principle (Ch. 2):** Visualise the data, look for correlations, discover patterns, and form hypotheses — all *before* touching the test set. The goal is to gain insights that will inform feature engineering and model selection.

---

## 0. Setup & Data

In [ ]:
from pathlib import Path
import sys, warnings

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams["figure.dpi"] = 120
warnings.filterwarnings("ignore")

FIGURES = PROJECT_ROOT / "reports" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

In [ ]:
from cardio_catch_disease.data import load_training_frame
from cardio_catch_disease.config import load_config
from sklearn.model_selection import train_test_split

config = load_config(PROJECT_ROOT / "configs" / "project.toml")
df = load_training_frame(config)

# Recreate the same split as Notebook 01
train_set, _ = train_test_split(df, test_size=0.20, random_state=42, stratify=df["cardio"])
df_train = train_set.copy()

# Clean blood pressure outliers before EDA
df_clean = df_train[
    (df_train["ap_hi"].between(70, 250)) &
    (df_train["ap_lo"].between(40, 150)) &
    (df_train["ap_hi"] > df_train["ap_lo"])
].copy()

# Engineered features for EDA
df_clean["age_years"]        = (df_clean["age"] / 365.25).round(1)
df_clean["bmi"]              = (df_clean["weight"] / (df_clean["height"] / 100) ** 2).round(1)
df_clean["pulse_pressure"]   = df_clean["ap_hi"] - df_clean["ap_lo"]
df_clean["high_bp"]          = ((df_clean["ap_hi"] >= 140) | (df_clean["ap_lo"] >= 90)).astype(int)
df_clean["bmi_category"]     = pd.cut(df_clean["bmi"], bins=[0, 18.5, 25, 30, 100],
                                       labels=["Underweight", "Normal", "Overweight", "Obese"])

print(f"Clean training set: {len(df_clean):,} rows (removed {len(df_train)-len(df_clean):,} outliers)")

## 1. Hypothesis: Age increases cardiovascular risk

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Age distribution by target
for label, color in [(0, "#2ecc71"), (1, "#e74c3c")]:
    df_clean[df_clean["cardio"] == label]["age_years"].hist(
        bins=40, ax=axes[0], alpha=0.6, color=color,
        label=["No Disease", "Disease"][label], density=True)
axes[0].set_title("Age Distribution by Cardiovascular Status", fontweight="bold")
axes[0].set_xlabel("Age (years)")
axes[0].set_ylabel("Density")
axes[0].legend()

# Disease rate by age group
df_clean["age_group"] = pd.cut(df_clean["age_years"], bins=range(29, 66, 5))
rate_by_age = df_clean.groupby("age_group")["cardio"].mean() * 100
rate_by_age.plot.bar(ax=axes[1], color="#e74c3c", edgecolor="white", rot=45)
axes[1].set_title("Disease Rate by Age Group (%)", fontweight="bold")
axes[1].set_ylabel("% with cardiovascular disease")
axes[1].axhline(50, color="gray", linestyle="--", alpha=0.5, label="50% baseline")
axes[1].legend()

plt.tight_layout()
plt.savefig(FIGURES / "02_age_vs_cardio.png", bbox_inches="tight")
plt.show()

print("\n→ CONFIRMED: Disease rate increases sharply with age.")

## 2. Hypothesis: High blood pressure is a strong risk factor

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Systolic BP boxplot
df_clean.boxplot(column="ap_hi", by="cardio", ax=axes[0],
                 boxprops=dict(color="#3498db"), medianprops=dict(color="red", linewidth=2))
axes[0].set_title("Systolic BP by Cardio Status")
axes[0].set_xlabel("Cardio (0=No, 1=Yes)")
axes[0].set_ylabel("mmHg")

# Diastolic BP boxplot
df_clean.boxplot(column="ap_lo", by="cardio", ax=axes[1],
                 boxprops=dict(color="#e74c3c"), medianprops=dict(color="navy", linewidth=2))
axes[1].set_title("Diastolic BP by Cardio Status")
axes[1].set_xlabel("Cardio (0=No, 1=Yes)")
axes[1].set_ylabel("mmHg")

# High BP flag disease rate
high_bp_rate = df_clean.groupby("high_bp")["cardio"].mean() * 100
high_bp_rate.rename({0: "Normal BP", 1: "Hypertension"}).plot.bar(
    ax=axes[2], color=["#2ecc71", "#e74c3c"], edgecolor="white", rot=0)
axes[2].set_title("Disease Rate: Normal vs High BP", fontweight="bold")
axes[2].set_ylabel("% with disease")
for bar in axes[2].patches:
    axes[2].annotate(f"{bar.get_height():.1f}%",
                     (bar.get_x() + bar.get_width()/2, bar.get_height()),
                     ha="center", va="bottom", fontsize=11)

plt.suptitle("Blood Pressure vs Cardiovascular Disease", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(FIGURES / "02_bp_vs_cardio.png", bbox_inches="tight")
plt.show()

print("\n→ CONFIRMED: Hypertension patients have a significantly higher disease rate.")

## 3. Hypothesis: BMI elevation correlates with cardiovascular risk

In [ ]:
bmi_valid = df_clean[df_clean["bmi"].between(10, 60)]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# BMI distributions
for label, color in [(0, "#2ecc71"), (1, "#e74c3c")]:
    bmi_valid[bmi_valid["cardio"] == label]["bmi"].hist(
        bins=40, ax=axes[0], alpha=0.6, color=color,
        label=["No Disease", "Disease"][label], density=True)
axes[0].set_title("BMI Distribution by Cardiovascular Status", fontweight="bold")
axes[0].set_xlabel("BMI (kg/m²)")
axes[0].axvline(25, color="gray", linestyle="--", alpha=0.6, label="Normal/Overweight cutoff")
axes[0].axvline(30, color="orange", linestyle="--", alpha=0.6, label="Overweight/Obese cutoff")
axes[0].legend(fontsize=8)

# Disease rate by BMI category
bmi_rate = bmi_valid.groupby("bmi_category", observed=True)["cardio"].mean() * 100
bmi_rate.plot.bar(ax=axes[1], color=["#3498db", "#2ecc71", "#e67e22", "#e74c3c"],
                  edgecolor="white", rot=0)
axes[1].set_title("Disease Rate by BMI Category", fontweight="bold")
axes[1].set_ylabel("% with cardiovascular disease")
for bar in axes[1].patches:
    axes[1].annotate(f"{bar.get_height():.1f}%",
                     (bar.get_x() + bar.get_width()/2, bar.get_height()),
                     ha="center", va="bottom", fontsize=10)

plt.tight_layout()
plt.savefig(FIGURES / "02_bmi_vs_cardio.png", bbox_inches="tight")
plt.show()

print("\n→ CONFIRMED: Disease rate increases progressively with BMI category.")

## 4. Cholesterol and Glucose Levels

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

level_labels = {1: "Normal", 2: "Above Normal", 3: "Well Above"}

for ax, col, title in [
    (axes[0], "cholesterol", "Cholesterol"),
    (axes[1], "gluc",        "Glucose"),
]:
    rate = df_clean.groupby(col)["cardio"].mean() * 100
    rate.index = [level_labels[i] for i in rate.index]
    rate.plot.bar(ax=ax, color=["#2ecc71", "#e67e22", "#e74c3c"], edgecolor="white", rot=0)
    ax.set_title(f"{title} Level vs Disease Rate", fontweight="bold")
    ax.set_ylabel("% with cardiovascular disease")
    for bar in ax.patches:
        ax.annotate(f"{bar.get_height():.1f}%",
                    (bar.get_x() + bar.get_width()/2, bar.get_height()),
                    ha="center", va="bottom", fontsize=10)

plt.tight_layout()
plt.savefig(FIGURES / "02_cholesterol_glucose.png", bbox_inches="tight")
plt.show()

print("\n→ CONFIRMED: Elevated cholesterol strongly predicts disease. Glucose has a weaker signal.")

## 5. Correlation Matrix

In [ ]:
num_cols = ["age_years", "height", "weight", "bmi", "ap_hi", "ap_lo",
            "pulse_pressure", "cholesterol", "gluc", "smoke", "alco", "active", "cardio"]
corr = df_clean[num_cols].corr()

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt=".2f", cmap="RdYlGn",
    center=0, vmin=-0.7, vmax=0.7, square=True, linewidths=0.5,
    cbar_kws={"shrink": 0.8}, ax=ax
)
ax.set_title("Correlation Matrix — Training Set Features", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(FIGURES / "02_correlation_matrix.png", bbox_inches="tight")
plt.show()

# Top correlations with target
target_corr = corr["cardio"].drop("cardio").sort_values(ascending=False)
print("\nTop feature correlations with cardio target:")
print(target_corr.round(3).to_string())

## 6. Feature vs Target — Violin Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
label_map = {0: "No Disease", 1: "Disease"}

for ax, col, label in [
    (axes[0], "age_years",     "Age (years)"),
    (axes[1], "bmi",           "BMI (kg/m²)"),
    (axes[2], "pulse_pressure","Pulse Pressure (mmHg)"),
]:
    plot_data = df_clean[[col, "cardio"]].copy()
    if col == "bmi":
        plot_data = plot_data[plot_data[col].between(10, 60)]
    plot_data["cardio_label"] = plot_data["cardio"].map(label_map)
    sns.violinplot(data=plot_data, x="cardio_label", y=col,
                   palette=["#2ecc71", "#e74c3c"], ax=ax, inner="quartile")
    ax.set_title(f"{label} by Cardio Status", fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel(label)

plt.suptitle("Feature Distributions by Cardiovascular Status", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(FIGURES / "02_violin_plots.png", bbox_inches="tight")
plt.show()

## 7. Lifestyle Factors — Do They Matter?

In [ ]:
lifestyle_cols = {"smoke": "Smoker", "alco": "Alcohol User", "active": "Physically Active"}

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (col, label) in zip(axes, lifestyle_cols.items()):
    rates = df_clean.groupby(col)["cardio"].mean() * 100
    rates.index = ["No", "Yes"]
    rates.plot.bar(ax=ax, color=["#3498db", "#e67e22"], edgecolor="white", rot=0)
    ax.set_title(f"{label} → Disease Rate", fontweight="bold")
    ax.set_ylabel("% with disease")
    ax.set_ylim(0, 80)
    for bar in ax.patches:
        ax.annotate(f"{bar.get_height():.1f}%",
                    (bar.get_x() + bar.get_width()/2, bar.get_height()),
                    ha="center", va="bottom", fontsize=11)

plt.suptitle("Lifestyle Factors vs Cardiovascular Disease Rate", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(FIGURES / "02_lifestyle_vs_cardio.png", bbox_inches="tight")
plt.show()

print("\n→ Smoking and alcohol have weak/unexpected signals.")
print("→ Physical activity shows a mild protective effect.")
print("→ These features likely add little predictive power individually.")

## 8. Scatter Plot: Age vs BMI coloured by Target

In [ ]:
sample = df_clean[df_clean["bmi"].between(10, 60)].sample(5000, random_state=42)

fig, ax = plt.subplots(figsize=(9, 6))
colors_map = {0: "#2ecc71", 1: "#e74c3c"}
for label, group in sample.groupby("cardio"):
    ax.scatter(group["age_years"], group["bmi"], c=colors_map[label],
               alpha=0.4, s=10, label=["No Disease", "Disease"][label])

ax.set_xlabel("Age (years)", fontsize=12)
ax.set_ylabel("BMI (kg/m²)", fontsize=12)
ax.set_title("Age vs BMI — coloured by Cardiovascular Status (n=5,000 sample)",
             fontsize=12, fontweight="bold")
ax.axhline(30, color="gray", linestyle="--", alpha=0.5, label="BMI=30 (obese threshold)")
ax.legend(markerscale=3, fontsize=10)
plt.tight_layout()
plt.savefig(FIGURES / "02_age_bmi_scatter.png", bbox_inches="tight")
plt.show()

## 9. EDA Summary — Confirmed Hypotheses

| Hypothesis | Status | Key Finding |
|---|---|---|
| Age increases cardiovascular risk | ✅ Confirmed | Disease rate rises sharply after 50 |
| High blood pressure is a strong predictor | ✅ Confirmed | Hypertensives have ~2× higher disease rate |
| BMI elevation correlates with risk | ✅ Confirmed | Obese patients have highest disease rate |
| Elevated cholesterol predicts risk | ✅ Confirmed | Well-above-normal group has highest rate |
| Glucose is a weaker predictor | ⚠️ Partial | Moderate separation only at highest level |
| Smoking / alcohol strongly predict risk | ❌ Rejected | Minimal signal — likely confounded |
| Physical activity is protective | ⚠️ Mild | Small protective effect observed |

**Feature importance ranking (pre-modelling intuition):**
> `ap_hi` ≈ `ap_lo` > `age_years` > `bmi` > `cholesterol` > `gluc` > `active` > `smoke` / `alco`

**Next → Notebook 03: Feature Engineering & Data Preparation**